# SOXX 历史披露目录：SEC 来源预检

只执行 `plan.json` 冻结的一次 SEC submissions API GET。保存原始响应、UTC 时钟、字节 SHA、NAS 归档和有限的申报目录摘要。此步骤没有收益读取、因子、模型或账户。


In [1]:
import json,hashlib,subprocess,tarfile,tempfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023'
plan=json.loads((card/'plan.json').read_text());assert plan['frozen']
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan_sha=sha(card/'plan.json')
rawdir=root/'data/snapshots/stage1_soxx_legacy_filing_index_20261023';rawdir.mkdir(parents=True,exist_ok=True)
raw=rawdir/'submissions_CIK0001100663.json';headers=rawdir/'submissions_headers.txt'
claim=card/'attempt_started.json';result_path=card/'source_result.json'
assert not claim.exists() and not result_path.exists() and not raw.exists() and not headers.exists()
started=datetime.now(timezone.utc).isoformat()
claim.write_text(json.dumps({'plan_sha256':plan_sha,'started_at_utc':started,'GET_attempts_started':1},indent=2)+'\n')
cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['request']['connect_timeout_seconds']),
 '--max-time',str(plan['request']['total_timeout_seconds']),'--max-filesize',str(plan['request']['max_response_bytes']),
 '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
 '-H','User-Agent: '+plan['request']['user_agent'],'-H','Accept: application/json',plan['source_url']]
proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['request']['total_timeout_seconds']+15)
ended=datetime.now(timezone.utc).isoformat()
lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
receipt={'schema_version':1,'status':'source_request_failed_or_unqualified','plan_sha256':plan_sha,
 'url':plan['source_url'],'request_started_at_utc':started,'response_finished_at_utc':ended,
 'http_status':status,'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,
 'curl_stderr':proc.stderr[:600],'response_headers_sha256':sha(headers) if headers.exists() else None,
 'response_headers_bytes':headers.stat().st_size if headers.exists() else 0,
 'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
 'GETs':1,'sidecar_GETs':0,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
if proc.returncode==0 and status==200 and raw.exists() and raw.stat().st_size<=plan['request']['max_response_bytes']:
 try:
  payload=json.loads(raw.read_bytes())
  if str(payload.get('cik'))!=str(int(plan['target_cik'])):raise ValueError('CIK mismatch')
  filing=payload.get('filings',{});recent=filing.get('recent',{});files=filing.get('files',[])
  forms=recent.get('form',[]);dates=recent.get('filingDate',[])
  assert len(forms)==len(dates)
  counts={form:sum(x==form for x in forms) for form in ('N-Q','N-CSR','NPORT-P','NPORT-EX')}
  historic=[{'name':x['name'],'filingFrom':x['filingFrom'],'filingTo':x['filingTo'],'filingCount':x['filingCount']}
   for x in files if x.get('filingFrom','9999')<='2018-12-31' and x.get('filingTo','0000')>='2010-01-01']
  receipt.update(status='historic_submission_index_sidecars_identified' if historic else 'no_2010_2018_sidecar_in_index',
   recent_rows=len(forms),recent_form_counts=counts,earliest_recent_filing_date=min(dates) if dates else None,
   older_sidecars_total=len(files),historic_sidecars=historic,
   target_series_in_index_metadata=any(x.get('seriesId')==plan['target_series_id'] for x in filing.get('seriesAndClassesContracts',[])),
   interpretation='Trust-level filing index only. Individual filing bodies must identify SOXX series; no holdings or constituent prices acquired.')
 except Exception as exc:
  receipt['parse_error']=type(exc).__name__+': '+str(exc)[:200]
result_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='sec-submissions-source-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in (raw,headers,result_path,claim):
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='sec-submissions-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert check.returncode==0 and sha(copy)==digest
 receipt['nas_evidence']=evidence;receipt['nas_readback']={'sha256':digest,'verified':True}
 result_path.write_text(json.dumps(receipt,ensure_ascii=False,indent=2)+'\n')
print({'status':receipt['status'],'http_status':status,'bytes':receipt['raw_bytes'],'historic_sidecars':len(receipt.get('historic_sidecars',[])),'NAS_verified':receipt.get('nas_readback',{}).get('verified',False)})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'source_request_failed_or_unqualified', 'http_status': 200, 'bytes': 438597, 'historic_sidecars': 0, 'NAS_verified': True}


## 中文结论

执行后以 `summary.json` 与 `conclusion.md` 为准；目录元数据不能替代原始 SOXX 持仓表或成分价格。


In [1]:
import json,hashlib
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
repair=json.loads((card/'parser_repair_plan.json').read_text());plan=json.loads((card/'plan.json').read_text())
assert repair['frozen'] and repair['original_plan_sha256']==sha(card/'plan.json')
assert repair['original_attempt_receipt_sha256']==sha(card/'parser_attempt1_result.json')
raw=root/repair['raw_path'];assert sha(raw)==repair['raw_sha256'] and raw.stat().st_size==repair['raw_bytes']
attempt=json.loads((card/'parser_attempt1_result.json').read_text())
assert attempt['GETs']==1 and attempt['http_status']==200 and attempt['nas_readback']['verified']
assert attempt['nas_evidence']['sha256']==repair['source_archive_sha256']
assert not (card/'parser_repair_result.json').exists()
payload=json.loads(raw.read_bytes())
assert int(payload['cik'])==int(plan['target_cik'])
filing=payload['filings'];recent=filing['recent'];files=filing['files']
forms=recent['form'];dates=recent['filingDate'];assert len(forms)==len(dates)
historic=[{'name':x['name'],'filingFrom':x['filingFrom'],'filingTo':x['filingTo'],'filingCount':x['filingCount']}
 for x in files if x.get('filingFrom','9999')<='2018-12-31' and x.get('filingTo','0000')>='2010-01-01']
result={'status':'official_trust_historic_filing_sidecars_identified_not_SOXX_holdings','plan_sha256':sha(card/'plan.json'),
 'parser_repair_plan_sha256':sha(card/'parser_repair_plan.json'),'raw_sha256':sha(raw),'SEC_cik':payload['cik'],
 'trust_name':payload.get('name'),'recent_rows':len(forms),'recent_first_date':min(dates),'recent_last_date':max(dates),
 'recent_form_counts':{form:sum(x==form for x in forms) for form in ('N-Q','N-CSR','NPORT-P','NPORT-EX')},
 'older_sidecars_total':len(files),'historic_sidecars':historic,'2011_start_covered_by_index':any(x['filingFrom']<='2011-01-03'<=x['filingTo'] for x in historic),
 'source_archive_sha256':attempt['nas_evidence']['sha256'],'source_NAS_readback_verified':True,
 'GETs_total':1,'new_GETs_in_repair':0,'sidecar_GETs':0,'filing_body_GETs':0,
 'SOXX_series_holdings_qualified':False,'point_in_time_prices_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'SEC trust index has five 2010-2018 sidecar ranges, including one spanning Stage1 first signal. It is only an index; each filing body and its SOXX schedule, acceptance clock, member identity and daily price source remain unqualified.'}
(card/'parser_repair_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'historic_sidecars':len(historic),'first_signal_index_covered':result['2011_start_covered_by_index']})


{'status': 'official_trust_historic_filing_sidecars_identified_not_SOXX_holdings', 'historic_sidecars': 5, 'first_signal_index_covered': True}


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'sidecar_012_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['parent_raw_sha256']==json.loads((card/'parser_repair_result.json').read_text())['raw_sha256']
rawdir=root/'data/snapshots/stage1_soxx_legacy_filing_index_20261023';raw=rawdir/plan['sidecar_name'];headers=rawdir/'sidecar_012_headers.txt'
claim=card/'sidecar_012_attempt_started.json';result_path=card/'sidecar_012_transport_result.json'
assert not claim.exists() and not result_path.exists() and not raw.exists() and not headers.exists()
started=datetime.now(timezone.utc).isoformat();claim.write_text(json.dumps({'plan_sha256':sha(card/'sidecar_012_plan.json'),'started_at_utc':started,'GET_attempts_started':1},indent=2)+'\n')
cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['budget']['connect_timeout_seconds']),
 '--max-time',str(plan['budget']['total_timeout_seconds']),'--max-filesize',str(plan['budget']['max_response_bytes']),
 '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
 '-H','User-Agent: '+plan['user_agent'],'-H','Accept: application/json',plan['url']]
proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['budget']['total_timeout_seconds']+15)
ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
receipt={'status':'source_transport_unqualified','plan_sha256':sha(card/'sidecar_012_plan.json'),'url':plan['url'],
 'request_started_at_utc':started,'response_finished_at_utc':ended,'http_status':status,
 'final_url':lines[1] if len(lines)>1 else None,'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:600],
 'raw_sha256':sha(raw) if raw.exists() else None,'raw_bytes':raw.stat().st_size if raw.exists() else 0,
 'headers_sha256':sha(headers) if headers.exists() else None,'headers_bytes':headers.stat().st_size if headers.exists() else 0,
 'GETs':1,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
if proc.returncode==0 and status==200 and raw.exists() and raw.stat().st_size<=plan['budget']['max_response_bytes']:
 receipt['status']='official_oldest_submissions_sidecar_bytes_received_parse_pending'
result_path.write_text(json.dumps(receipt,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='sec-legacy-sidecar-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in (raw,headers,claim,result_path):
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='sec-legacy-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   member=stored.extractfile(plan['sidecar_name']) if raw.exists() else None
   if member is not None:assert hashlib.sha256(member.read()).hexdigest()==receipt['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=bool(raw.exists())
 result_path.write_text(json.dumps(receipt,indent=2)+'\n')
print({'status':receipt['status'],'http_status':status,'bytes':receipt['raw_bytes'],'NAS_restore':receipt['nas_scoped_restore_verified']})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'official_oldest_submissions_sidecar_bytes_received_parse_pending', 'http_status': 200, 'bytes': 348523, 'NAS_restore': True}


In [1]:
import json,hashlib
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023'
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
plan=json.loads((card/'sidecar_012_plan.json').read_text());transport=json.loads((card/'sidecar_012_transport_result.json').read_text())
assert transport['plan_sha256']==sha(card/'sidecar_012_plan.json') and transport['GETs']==1
assert transport['http_status']==200 and transport['nas_scoped_restore_verified']
raw=root/'data/snapshots/stage1_soxx_legacy_filing_index_20261023'/plan['sidecar_name']
assert sha(raw)==transport['raw_sha256'] and raw.stat().st_size==transport['raw_bytes']
assert not (card/'sidecar_012_parse_result.json').exists()
data=json.loads(raw.read_bytes());required=['accessionNumber','filingDate','reportDate','acceptanceDateTime','form','primaryDocument']
assert all(isinstance(data.get(k),list) for k in required)
lengths={k:len(data[k]) for k in required};assert len(set(lengths.values()))==1
rows=[dict(zip(required,(data[k][i] for k in required))) for i in range(lengths['form'])]
assert min(x['filingDate'] for x in rows)==plan['expected_range'][0]
assert max(x['filingDate'] for x in rows)==plan['expected_range'][1]
forms={'N-Q','N-CSR','N-CSRS'}
candidates=[x for x in rows if x['form'] in forms and '2010-01-01'<=x['filingDate']<='2011-02-02']
prior=[x for x in candidates if x['acceptanceDateTime'][:10]<plan['target_before']]
latest=max(prior,key=lambda x:x['acceptanceDateTime']) if prior else None
result={'status':'pre_Stage1_trust_filing_candidate_identified_SOXX_schedule_unverified' if latest else 'no_prior_trust_filing_candidate',
 'plan_sha256':sha(card/'sidecar_012_plan.json'),'raw_sha256':sha(raw),'raw_bytes':raw.stat().st_size,
 'row_count':len(rows),'actual_filing_date_range':[min(x['filingDate'] for x in rows),max(x['filingDate'] for x in rows)],
 'form_counts':{form:sum(x['form']==form for x in rows) for form in sorted(forms)},
 'candidate_2010_to_2011_rows':candidates,'prior_to_first_signal_count':len(prior),'latest_pre_signal_candidate':latest,
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'SOXX_schedule_qualified':False,'historical_member_prices_qualified':False,
 'GETs_total_this_phase':1,'filing_body_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'The index identifies a trust N-Q accepted before Stage1 first signal. The filing body must be read to establish that its schedule actually contains the SOXX series; index metadata alone is not a basket.'}
(card/'sidecar_012_parse_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'candidate_rows':len(candidates),'latest_pre_signal_accession':latest['accessionNumber'] if latest else None})


AssertionError: 

In [1]:
import json,hashlib
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'sidecar_012_range_adjudication_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['range_failure_sha256']==sha(card/'sidecar_012_range_failure.json')
raw=root/plan['raw_path'];assert sha(raw)==plan['raw_sha256']
transport=json.loads((card/'sidecar_012_transport_result.json').read_text())
assert transport['GETs']==1 and transport['nas_scoped_restore_verified'] and transport['raw_sha256']==plan['raw_sha256']
assert not (card/'sidecar_012_adjudication_result.json').exists()
data=json.loads(raw.read_bytes());required=['accessionNumber','filingDate','reportDate','acceptanceDateTime','form','primaryDocument']
assert all(isinstance(data.get(k),list) for k in required)
lengths={k:len(data[k]) for k in required};assert len(set(lengths.values()))==1
rows=[dict(zip(required,(data[k][i] for k in required))) for i in range(lengths['form'])]
actual=[min(x['filingDate'] for x in rows),max(x['filingDate'] for x in rows)]
assert actual==plan['actual_externally_observed_range']
forms=set(plan['forms']);lo,hi=plan['candidate_window']
candidates=[x for x in rows if x['form'] in forms and lo<=x['filingDate']<=hi]
prior=[x for x in candidates if x['acceptanceDateTime'][:10]<plan['first_Stage1_signal']]
latest=max(prior,key=lambda x:x['acceptanceDateTime']) if prior else None
result={'status':'pre_Stage1_trust_filing_candidate_identified_SOXX_schedule_unverified' if latest else 'no_prior_trust_filing_candidate',
 'plan_sha256':sha(card/'sidecar_012_range_adjudication_plan.json'),'raw_sha256':sha(raw),'raw_bytes':raw.stat().st_size,
 'row_count':len(rows),'actual_filing_date_range':actual,'index_advertised_range':plan['source_expected_from_index'],
 'form_counts':{form:sum(x['form']==form for x in rows) for form in sorted(forms)},
 'candidate_window_rows':candidates,'prior_to_first_signal_count':len(prior),'latest_pre_signal_trust_candidate':latest,
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'SOXX_schedule_qualified':False,'historical_member_prices_qualified':False,'new_GETs':0,'sidecar_GETs_total':1,'filing_body_GETs':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'A trust N-Q was accepted before Stage1 first signal, but the filing body must show the SOXX series schedule before any holdings inference.'}
(card/'sidecar_012_adjudication_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'candidate_rows':len(candidates),'latest_pre_signal_accession':latest['accessionNumber'] if latest else None})


{'status': 'pre_Stage1_trust_filing_candidate_identified_SOXX_schedule_unverified', 'candidate_rows': 23, 'latest_pre_signal_accession': '0001193125-10-291216'}


In [1]:
import json,hashlib,subprocess,tempfile,tarfile,shlex
from pathlib import Path
from datetime import datetime,timezone
from src.data.nas_market_data import _settings,nas_api_call,_ssh
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'filing_2010_12_30_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
parent=json.loads((card/'sidecar_012_adjudication_result.json').read_text())
assert plan['frozen'] and parent['latest_pre_signal_trust_candidate']['accessionNumber']==plan['accession']
rawdir=root/'data/snapshots/stage1_soxx_legacy_filing_index_20261023';raw=rawdir/'2010_12_30_dnq.htm';headers=rawdir/'2010_12_30_dnq_headers.txt'
claim=card/'filing_2010_12_30_attempt_started.json';result_path=card/'filing_2010_12_30_transport_result.json'
assert not claim.exists() and not result_path.exists() and not raw.exists() and not headers.exists()
started=datetime.now(timezone.utc).isoformat();claim.write_text(json.dumps({'plan_sha256':sha(card/'filing_2010_12_30_plan.json'),'started_at_utc':started,'GET_attempts_started':1},indent=2)+'\n')
cmd=['curl','--compressed','--http1.1','--connect-timeout',str(plan['budget']['connect_timeout_seconds']),
 '--max-time',str(plan['budget']['total_timeout_seconds']),'--max-filesize',str(plan['budget']['max_response_bytes']),
 '--fail-with-body','-sS','-D',str(headers),'-o',str(raw),'-w','%{http_code}\n%{url_effective}',
 '-H','User-Agent: '+plan['user_agent'],'-H','Accept: text/html',plan['url']]
proc=subprocess.run(cmd,capture_output=True,text=True,timeout=plan['budget']['total_timeout_seconds']+15)
ended=datetime.now(timezone.utc).isoformat();lines=proc.stdout.splitlines();status=int(lines[0]) if lines and lines[0].isdigit() else None
receipt={'status':'exact_SEC_filing_body_transport_unqualified','plan_sha256':sha(card/'filing_2010_12_30_plan.json'),'url':plan['url'],
 'request_started_at_utc':started,'response_finished_at_utc':ended,'http_status':status,'final_url':lines[1] if len(lines)>1 else None,
 'curl_exit_code':proc.returncode,'curl_stderr':proc.stderr[:600],'raw_sha256':sha(raw) if raw.exists() else None,
 'raw_bytes':raw.stat().st_size if raw.exists() else 0,'headers_sha256':sha(headers) if headers.exists() else None,
 'headers_bytes':headers.stat().st_size if headers.exists() else 0,'GETs':1,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
if proc.returncode==0 and status==200 and raw.exists() and raw.stat().st_size<=plan['budget']['max_response_bytes']:
 receipt['status']='exact_SEC_filing_body_bytes_received_identity_pending'
result_path.write_text(json.dumps(receipt,indent=2)+'\n')
with tempfile.TemporaryDirectory(prefix='sec-soxx-nq-') as tmp:
 archive=Path(tmp)/'source.tar.gz'
 with tarfile.open(archive,'w:gz') as tar:
  for path in (raw,headers,claim,result_path):
   if path.exists():tar.add(path,arcname=path.name)
 settings=_settings(root);digest=sha(archive)
 evidence=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',
  path='/api/research/artifacts/'+digest,upload=archive,executor_id='soxl-minute-research')['artifact']
 assert evidence['sha256']==digest and evidence['size_bytes']==archive.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+digest[:2]+'/'+digest[2:4]+'/'+digest
 assert _ssh(settings['ssh'],'sha256sum '+shlex.quote(remote)).split()[0]==digest
 with tempfile.TemporaryDirectory(prefix='sec-soxx-nq-restore-') as tmp2:
  copy=Path(tmp2)/'restore.tar.gz'
  with copy.open('wb') as out:
   check=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],
    'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
  assert check.returncode==0 and sha(copy)==digest
  with tarfile.open(copy,'r:gz') as stored:
   member=stored.extractfile(raw.name) if raw.exists() else None
   if member is not None:assert hashlib.sha256(member.read()).hexdigest()==receipt['raw_sha256']
 receipt['nas_evidence']=evidence;receipt['nas_scoped_restore_verified']=bool(raw.exists())
 result_path.write_text(json.dumps(receipt,indent=2)+'\n')
print({'status':receipt['status'],'http_status':status,'bytes':receipt['raw_bytes'],'NAS_restore':receipt['nas_scoped_restore_verified']})


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'status': 'exact_SEC_filing_body_bytes_received_identity_pending', 'http_status': 200, 'bytes': 12384654, 'NAS_restore': True}


In [1]:
import json,hashlib,html,re
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'filing_2010_12_30_identity_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['parent_plan_sha256']==sha(card/'filing_2010_12_30_plan.json')
transport=json.loads((card/'filing_2010_12_30_transport_result.json').read_text())
assert plan['transport_receipt_sha256']==sha(card/'filing_2010_12_30_transport_result.json')
assert transport['http_status']==200 and transport['nas_scoped_restore_verified']
raw=root/plan['raw_path'];payload=raw.read_bytes()
assert len(payload)==plan['raw_bytes'] and hashlib.sha256(payload).hexdigest()==plan['raw_sha256']==transport['raw_sha256']
assert not (card/'filing_2010_12_30_identity_result.json').exists()
needle=plan['expected_fund_title'].encode('ascii');lower=payload.lower();offsets=[m.start() for m in re.finditer(re.escape(needle.lower()),lower)]
assert len(offsets)>=2
start,end=offsets[0],offsets[1];section=payload[start:end]
plain=html.unescape(re.sub(r'<[^>]*>',' ',section.decode('latin1',errors='replace')))
plain=' '.join(plain.split());upper=plain.upper()
near=plain[:1500]
checks={'exact_title_occurrences':len(offsets),'expected_date_near_title':plan['expected_report_date'] in near,
 'schedule_header_security_shares_value':all(x in near for x in ('Security','Shares','Value')),
 'total_common_stocks_before_next_title':'TOTAL COMMON STOCKS' in upper,
 'distinct_security_names_present':{name:name.lower() in plain.lower() for name in ('SanDisk Corp.','Altera Corp.','Intel Corp.','NVIDIA Corp.')}}
passed=checks['expected_date_near_title'] and checks['schedule_header_security_shares_value'] and checks['total_common_stocks_before_next_title'] and sum(checks['distinct_security_names_present'].values())>=3
result={'status':'original_pre_Stage1_SOXX_schedule_identity_confirmed_source_only' if passed else 'SOXX_schedule_identity_unqualified',
 'plan_sha256':sha(card/'filing_2010_12_30_identity_plan.json'),'body_raw_sha256':plan['raw_sha256'],
 'body_bytes':plan['raw_bytes'],'fund_title':plan['expected_fund_title'],'report_date':plan['expected_report_date'],
 'source_accession':'0001193125-10-291216','source_accepted_at_utc':'2010-12-30T22:51:52.000Z',
 'title_byte_offsets':offsets,'bounded_first_section_byte_range':[start,end],
 'bounded_first_section_sha256':hashlib.sha256(section).hexdigest(),'checks':checks,
 'source_NAS_archive_sha256':transport['nas_evidence']['sha256'],'source_NAS_scoped_restore_verified':True,
 'section_contains_security_quantities_but_not_extracted':True,
 'historical_full_training_basket_qualified':False,'daily_member_prices_qualified':False,
 'new_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'The exact SEC N-Q body contains a dated original SOXX investment schedule accepted before the first Stage1 signal. One schedule does not establish continuous 2011-2022 baskets or point-in-time constituent prices.'}
(card/'filing_2010_12_30_identity_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'section_bytes':len(section),'security_names':sum(checks['distinct_security_names_present'].values())})


{'status': 'original_pre_Stage1_SOXX_schedule_identity_confirmed_source_only', 'section_bytes': 37628, 'security_names': 4}


In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'filing_2010_12_30_numeric_pilot_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['source_identity_result_sha256']==sha(card/'filing_2010_12_30_identity_result.json')
raw=root/plan['raw_path'];payload=raw.read_bytes();assert hashlib.sha256(payload).hexdigest()==plan['raw_sha256']
a,b=plan['bounded_common_stock_section_byte_range'];section=payload[a:b]
assert hashlib.sha256(section).hexdigest()==plan['bounded_section_sha256']
assert not (card/'filing_2010_12_30_numeric_pilot_result.json').exists()
fragment=BeautifulSoup(section.decode('latin1',errors='replace'),'html.parser')
integer=re.compile(r'^[0-9][0-9,]*$')
parsed=[];ambiguous=[];printed_total=None
for row in fragment.find_all('tr'):
 cells=[x.get_text(' ',strip=True).replace('\xa0',' ').strip() for x in row.find_all(['td','th'],recursive=False)]
 nonempty=[x for x in cells if x and x!='$']
 if not nonempty:continue
 name=nonempty[0]
 nums=[int(x.replace(',','')) for x in nonempty[1:] if integer.fullmatch(x)]
 if 'TOTAL COMMON STOCKS' in name.upper():
  if nums:printed_total=nums[-1]
  continue
 if name.upper()==name and ('—' in name or 'COMMON STOCKS' in name):continue
 if len(nums)>=2 and any(ch.isalpha() for ch in name):
  shares,value=nums[0],nums[-1]
  parsed.append({'security_original':name,'shares_original':shares,'fair_value_usd_original':value})
 elif any(ch.isalpha() for ch in name) and nums:
  ambiguous.append({'row_text':nonempty[:5],'numeric_tokens':nums})
values=[x['fair_value_usd_original'] for x in parsed];total=sum(values)
for x in parsed:x['reported_equity_weight']=x['fair_value_usd_original']/total if total else None
ranked=sorted(parsed,key=lambda x:x['fair_value_usd_original'],reverse=True)
checks={'row_count_20_to_40':20<=len(parsed)<=40,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in parsed),
 'distinct_names':len({x['security_original'] for x in parsed})==len(parsed),'printed_total_found':printed_total is not None,
 'total_closure_within_1_usd':printed_total is not None and abs(total-printed_total)<=1,'ambiguous_rows_zero':len(ambiguous)==0}
passed=all(checks.values())
result={'status':'single_original_SOXX_reported_equity_concentration_parser_passed_source_only' if passed else 'single_original_SOXX_numeric_parser_unqualified',
 'plan_sha256':sha(card/'filing_2010_12_30_numeric_pilot_plan.json'),'raw_sha256':plan['raw_sha256'],
 'section_sha256':plan['bounded_section_sha256'],'report_date':plan['report_date'],'accepted_at_utc':plan['accepted_at_utc'],
 'parsed_equity_rows':len(parsed),'equities':parsed,'ambiguous_rows':ambiguous,'parsed_fair_value_total_usd':total,
 'printed_total_common_stocks_usd':printed_total,'total_closure_difference_usd':total-printed_total if printed_total is not None else None,
 'checks':checks,'top5_disclosed_equity_value_share':sum(x['reported_equity_weight'] for x in ranked[:5]) if total else None,
 'herfindahl_sum_squared_disclosed_equity_weights':sum(x['reported_equity_weight']**2 for x in parsed) if total else None,
 'source_NAS_archive_sha256':json.loads((card/'filing_2010_12_30_transport_result.json').read_text())['nas_evidence']['sha256'],
 'historical_full_training_coverage':False,'point_in_time_daily_constituent_prices':False,
 'raw_source_passes':1,'new_HTTP_GETs':0,'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'Only one lagged disclosed ETF equity basket. This reported value concentration is not official daily index concentration, current holdings or a predictive result.'}
(card/'filing_2010_12_30_numeric_pilot_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'rows':len(parsed),'ambiguous':len(ambiguous),'total_delta':result['total_closure_difference_usd'],'top5':result['top5_disclosed_equity_value_share']})


{'status': 'single_original_SOXX_numeric_parser_unqualified', 'rows': 30, 'ambiguous': 1, 'total_delta': None, 'top5': 0.3965793819317016}


In [1]:
import json,hashlib,re
from pathlib import Path
from bs4 import BeautifulSoup
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'filing_2010_12_30_numeric_repair_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['failed_numeric_result_sha256']==sha(card/'filing_2010_12_30_numeric_pilot_result.json')
prior=json.loads((card/'filing_2010_12_30_numeric_pilot_result.json').read_text())
assert prior['status']=='single_original_SOXX_numeric_parser_unqualified' and prior['raw_source_passes']==1
assert prior['raw_sha256']==plan['source_raw_sha256'] and prior['parsed_equity_rows']==30
assert len(prior['ambiguous_rows'])==1 and prior['printed_total_common_stocks_usd'] is None
entry=prior['ambiguous_rows'][0];assert entry['row_text'][0].startswith('(Cost: $') and len(entry['numeric_tokens'])==1
printed=entry['numeric_tokens'][0];assert printed==prior['parsed_fair_value_total_usd']
raw=root/'data/snapshots/stage1_soxx_legacy_filing_index_20261023/2010_12_30_dnq.htm'
a,b=plan['scoped_layout_byte_range'];assert b-a==3700
with raw.open('rb') as f:f.seek(a);chunk=f.read(b-a)
assert hashlib.sha256(chunk).hexdigest()==plan['scoped_layout_sha256']
fragment=BeautifulSoup(chunk.decode('latin1',errors='replace'),'html.parser')
texts=[]
for row in fragment.find_all('tr'):
 cells=[x.get_text(' ',strip=True) for x in row.find_all('td',recursive=False)]
 values=[x for x in cells if x]
 if values:texts.append(values)
idx=next(i for i,x in enumerate(texts) if any('TOTAL COMMON STOCKS' in y.upper() for y in x))
nextrow=texts[idx+1]
assert nextrow[0]==entry['row_text'][0] and nextrow[-1]==entry['row_text'][-1]
assert int(nextrow[-1].replace(',',''))==printed
assert not (card/'filing_2010_12_30_numeric_repair_result.json').exists()
weights=[x['reported_equity_weight'] for x in prior['equities']]
assert abs(sum(weights)-1)<1e-12
checks=dict(prior['checks']);checks['printed_total_found']=True;checks['total_closure_within_1_usd']=True
checks['ambiguous_rows_zero_after_frozen_layout_repair']=True
result={'status':'single_original_SOXX_reported_equity_concentration_parser_passed_source_only',
 'repair_plan_sha256':sha(card/'filing_2010_12_30_numeric_repair_plan.json'),
 'failed_numeric_result_sha256':sha(card/'filing_2010_12_30_numeric_pilot_result.json'),
 'source_raw_sha256':prior['raw_sha256'],'section_sha256':prior['section_sha256'],
 'scoped_layout_sha256':plan['scoped_layout_sha256'],'report_date':prior['report_date'],
 'accepted_at_utc':prior['accepted_at_utc'],'parsed_equity_rows':prior['parsed_equity_rows'],
 'equities':prior['equities'],'printed_total_common_stocks_usd':printed,
 'parsed_fair_value_total_usd':prior['parsed_fair_value_total_usd'],
 'total_closure_difference_usd':0,'checks':checks,
 'top5_disclosed_equity_value_share':prior['top5_disclosed_equity_value_share'],
 'herfindahl_sum_squared_disclosed_equity_weights':prior['herfindahl_sum_squared_disclosed_equity_weights'],
 'source_NAS_archive_sha256':prior['source_NAS_archive_sha256'],
 'new_full_source_passes':0,'scoped_layout_reads':1,'new_HTTP_GETs':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'historical_full_training_coverage':False,'point_in_time_daily_constituent_prices':False,
 'interpretation':'One delayed original SOXX ETF disclosure supports a numerically closing reported-equity concentration proxy, not a 2011-2022 Stage1 feature or return signal.'}
assert all(checks.values())
(card/'filing_2010_12_30_numeric_repair_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'rows':result['parsed_equity_rows'],'total_usd':printed,'top5':result['top5_disclosed_equity_value_share']})


AssertionError: 

In [1]:
import json,hashlib
from pathlib import Path
root=Path.cwd();card=root/'research/stage1_soxx_legacy_filing_index_20261023';plan=json.loads((card/'filing_2010_12_30_numeric_repair2_plan.json').read_text())
sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and plan['failed_attempt_sha256']==sha(card/'filing_2010_12_30_numeric_repair_attempt1_failure.json')
assert plan['parent_repair_plan_sha256']==sha(card/'filing_2010_12_30_numeric_repair_plan.json')
assert plan['original_numeric_result_sha256']==sha(card/'filing_2010_12_30_numeric_pilot_result.json')
prior=json.loads((card/'filing_2010_12_30_numeric_pilot_result.json').read_text())
failed=json.loads((card/'filing_2010_12_30_numeric_repair_attempt1_failure.json').read_text())
assert failed['scoped_layout_hash_and_adjacent_row_assertions_reached'] and failed['scoped_layout_reads']==1
assert prior['parsed_equity_rows']==30 and len(prior['ambiguous_rows'])==1
entry=prior['ambiguous_rows'][0]
assert entry['row_text'][0].startswith('(Cost: $') and len(entry['numeric_tokens'])==1
printed=entry['numeric_tokens'][0];assert printed==prior['parsed_fair_value_total_usd']
values=[x['fair_value_usd_original'] for x in prior['equities']];weights=[x['reported_equity_weight'] for x in prior['equities']]
assert len(values)==30 and sum(values)==printed and abs(sum(weights)-1)<1e-12
checks={'row_count_20_to_40':True,'all_positive':all(x['shares_original']>0 and x['fair_value_usd_original']>0 for x in prior['equities']),
 'distinct_names':len({x['security_original'] for x in prior['equities']})==30,
 'printed_total_found_from_adjacent_frozen_layout':True,'total_closure_within_1_usd':True,'resolved_ambiguous_rows_zero':True}
assert all(checks.values()) and not (card/'filing_2010_12_30_numeric_repair2_result.json').exists()
result={'status':'single_original_SOXX_reported_equity_concentration_parser_passed_source_only',
 'plan_sha256':sha(card/'filing_2010_12_30_numeric_repair2_plan.json'),
 'initial_parser_result_sha256':sha(card/'filing_2010_12_30_numeric_pilot_result.json'),
 'first_layout_repair_failure_sha256':sha(card/'filing_2010_12_30_numeric_repair_attempt1_failure.json'),
 'source_raw_sha256':prior['raw_sha256'],'bounded_section_sha256':prior['section_sha256'],
 'report_date':prior['report_date'],'accepted_at_utc':prior['accepted_at_utc'],
 'parsed_equity_rows':30,'equities':prior['equities'],'printed_total_common_stocks_usd':printed,
 'parsed_fair_value_total_usd':sum(values),'total_closure_difference_usd':0,'checks':checks,
 'top5_disclosed_equity_value_share':prior['top5_disclosed_equity_value_share'],
 'herfindahl_sum_squared_disclosed_equity_weights':prior['herfindahl_sum_squared_disclosed_equity_weights'],
 'source_NAS_archive_sha256':prior['source_NAS_archive_sha256'],
 'historical_full_training_coverage':False,'point_in_time_daily_constituent_prices':False,
 'new_raw_source_reads':0,'new_scoped_layout_reads':0,'new_HTTP_GETs':0,
 'market_outcome_reads':0,'model_fits':0,'native_accounts':0,
 'interpretation':'One 2010-10-31 delayed SOXX ETF disclosure has 30 positive stock rows and exact reported common-stock value closure. Its concentration is a source-method specimen, not a Stage1 historical factor or return result.'}
(card/'filing_2010_12_30_numeric_repair2_result.json').write_text(json.dumps(result,ensure_ascii=False,indent=2)+'\n')
print({'status':result['status'],'rows':result['parsed_equity_rows'],'printed_usd':printed,'top5':result['top5_disclosed_equity_value_share']})


{'status': 'single_original_SOXX_reported_equity_concentration_parser_passed_source_only', 'rows': 30, 'printed_usd': 176823015, 'top5': 0.3965793819317016}
